# eKYC Document Fraud Detector — Training Notebook

**How to use:**
1. Go to `Runtime → Change runtime type → T4 GPU` (free)
2. Run each cell top to bottom
3. At the end, download `doc_fraud_detector.onnx` and place it in your project at `models/doc_fraud/`

**Estimated time:** ~10 minutes on T4 GPU

## Step 1 — Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

## Step 2 — Install dependencies

In [ ]:
!pip install -q opencv-python-headless face-recognition

## Step 3 — Upload your dataset

Create the two zip files on your Mac terminal:

```bash
cd /Users/chilanhouthnitvongkhay/Desktop/ekyc-system-laligence

# Zip real documents (no Mac metadata files)
zip -rj real_documents.zip data/real_documents/ -x "*/.*" -x "__MACOSX"

# Zip random photos (no Mac metadata files)
zip -rj random_photos.zip data/random_photos/ -x "*/.*" -x "__MACOSX"
```

Then upload both zips using the file picker below.

In [ ]:
from google.colab import files
import zipfile, os, shutil
from pathlib import Path

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

def extract_images_from_zip(zip_path, out_dir):
    """Extract zip and flatten all images into out_dir regardless of nested folders."""
    os.makedirs(out_dir, exist_ok=True)
    tmp = out_dir + '_tmp'
    os.makedirs(tmp, exist_ok=True)

    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(tmp)

    # Collect ALL images recursively from extracted folder
    all_imgs = [p for p in Path(tmp).rglob('*') if p.suffix.lower() in IMG_EXTS]
    print(f'  Found {len(all_imgs)} images inside zip')

    for p in all_imgs:
        dest = os.path.join(out_dir, p.name)
        # Avoid name collisions
        if os.path.exists(dest):
            dest = os.path.join(out_dir, f'{p.stem}_{p.parent.name}{p.suffix}')
        shutil.move(str(p), dest)

    shutil.rmtree(tmp)
    return len(list(Path(out_dir).glob('*')))

print('Upload real_documents.zip and random_photos.zip')
uploaded = files.upload()

for fname in uploaded:
    print(f'\nProcessing {fname} ...')
    if 'real' in fname.lower():
        n = extract_images_from_zip(fname, 'data/real_documents')
        print(f'  ✅ real_documents: {n} images ready')
    elif 'photo' in fname.lower() or 'random' in fname.lower():
        n = extract_images_from_zip(fname, 'data/random_photos')
        print(f'  ✅ random_photos: {n} images ready')
    else:
        print(f'  ⚠️  Could not identify "{fname}" — rename it to include "real" or "photo" and re-upload')

print(f'\nreal_documents: {len(list(Path("data/real_documents").glob("*")))} images')
print(f'random_photos:  {len(list(Path("data/random_photos").glob("*")))} images')

## Step 4 — Generate fake document samples

In [ ]:
import cv2
import numpy as np
import random
from pathlib import Path

random.seed(42)
np.random.seed(42)

def _jpeg_roundtrip(image, quality):
    bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    _, buf = cv2.imencode('.jpg', bgr, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return cv2.cvtColor(cv2.imdecode(buf, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)

def fake_clone_stamp(image):
    h, w = image.shape[:2]
    ph = random.randint(h // 8, h // 3)
    pw = random.randint(w // 8, w // 3)
    sy, sx = random.randint(0, h - ph), random.randint(0, w - pw)
    patch = image[sy:sy+ph, sx:sx+pw].copy()
    for _ in range(20):
        dy, dx = random.randint(0, h - ph), random.randint(0, w - pw)
        if abs(dy - sy) > ph // 2 or abs(dx - sx) > pw // 2:
            break
    fake = image.copy()
    fake[dy:dy+ph, dx:dx+pw] = patch
    mask = np.zeros((h, w), dtype=np.float32)
    mask[dy:dy+ph, dx:dx+pw] = 1.0
    k = max(3, min(ph, pw) // 6) | 1
    mask = cv2.GaussianBlur(mask, (k, k), 0)[:, :, None]
    fake = (fake * mask + image * (1 - mask)).astype(np.uint8)
    return _jpeg_roundtrip(fake, random.randint(70, 90))

def fake_region_splice(image, donors):
    if not donors: return None
    donor = random.choice(donors)
    h, w = image.shape[:2]
    dh, dw = donor.shape[:2]
    ph, pw = random.randint(h//6, h//2), random.randint(w//6, w//2)
    if dh < ph or dw < pw: return None
    sy, sx = random.randint(0, dh - ph), random.randint(0, dw - pw)
    patch = cv2.resize(donor[sy:sy+ph, sx:sx+pw], (pw, ph))
    dy, dx = random.randint(0, h - ph), random.randint(0, w - pw)
    fake = image.copy()
    fake[dy:dy+ph, dx:dx+pw] = patch
    return _jpeg_roundtrip(fake, random.randint(65, 85))

def load_rgb(path):
    img = cv2.imread(str(path))
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None else None

FAKE_DIR = Path('data/fake_documents')
FAKE_DIR.mkdir(parents=True, exist_ok=True)

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

# Use rglob to find images in any subdirectory
real_paths  = [p for p in Path('data/real_documents').rglob('*') if p.suffix.lower() in IMG_EXTS]
donor_paths = [p for p in Path('data/random_photos').rglob('*')  if p.suffix.lower() in IMG_EXTS]

print(f'Found {len(real_paths)} real document images')
print(f'Found {len(donor_paths)} random photo images')

if len(real_paths) == 0:
    raise ValueError('❌ No images found in data/real_documents/ — check your zip was uploaded and extracted correctly')
if len(donor_paths) == 0:
    raise ValueError('❌ No images found in data/random_photos/ — check your zip was uploaded and extracted correctly')

donor_imgs = [img for p in random.sample(donor_paths, min(200, len(donor_paths)))
              if (img := load_rgb(p)) is not None]

N_FAKE = len(real_paths)
techniques = [fake_clone_stamp, fake_region_splice]
generated = 0

for i in range(N_FAKE * 3):
    if generated >= N_FAKE: break
    img = load_rgb(random.choice(real_paths))
    if img is None: continue
    fn = random.choice(techniques)
    result = fn(img) if fn is fake_clone_stamp else fn(img, donor_imgs)
    if result is None: continue
    bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(FAKE_DIR / f'fake_{generated:05d}.jpg'), bgr, [cv2.IMWRITE_JPEG_QUALITY, 92])
    generated += 1
    if generated % 300 == 0:
        print(f'  {generated}/{N_FAKE} fake samples generated ...')

print(f'\n✅ Generated {generated} fake document samples → {FAKE_DIR}')

## Step 5 — Train EfficientNet-B0 classifier

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import EfficientNet_B0_Weights
from PIL import Image, UnidentifiedImageError

CLASSES   = ['real_document', 'not_document', 'fake_document']
IMG_SIZE  = 224
EPOCHS    = 30
BATCH     = 32
LR        = 1e-4
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

# ── Dataset ──
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class DocDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
            return self.transform(img), label
        except (UnidentifiedImageError, Exception):
            # Return a blank image for corrupted/unreadable files
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), color=0)
            return self.transform(img), label

img_exts = {'.jpg', '.jpeg', '.png', '.bmp'}

def collect_images(folder, label):
    return [
        (p, label)
        for p in Path(folder).rglob('*')
        if p.suffix.lower() in img_exts
        and not p.name.startswith('._')   # skip Mac metadata files
        and not p.name.startswith('.')    # skip all hidden files
        and p.stat().st_size > 1024       # skip files smaller than 1 KB
    ]

all_samples = (
    collect_images('data/real_documents', 0) +
    collect_images('data/random_photos',  1) +
    collect_images('data/fake_documents', 2)
)
random.shuffle(all_samples)
split = int(len(all_samples) * 0.85)
train_ds = DocDataset(all_samples[:split], train_tf)
val_ds   = DocDataset(all_samples[split:], val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')
print(f'  real_document: {sum(1 for _,l in all_samples if l==0)}')
print(f'  not_document:  {sum(1 for _,l in all_samples if l==1)}')
print(f'  fake_document: {sum(1 for _,l in all_samples if l==2)}')

# ── Model ──
model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(nn.Dropout(p=0.4), nn.Linear(in_features, 3))
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# ── Training loop ──
best_val_acc = 0.0
best_state   = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum = correct = total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * images.size(0)
        correct  += (out.argmax(1) == labels).sum().item()
        total    += images.size(0)
    scheduler.step()

    model.eval()
    vc = vt = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            vc += (model(images).argmax(1) == labels).sum().item()
            vt += images.size(0)

    train_acc = correct / max(total, 1)
    val_acc   = vc / max(vt, 1)
    print(f'Epoch {epoch:3d}/{EPOCHS}  loss={loss_sum/total:.4f}  train_acc={train_acc:.3f}  val_acc={val_acc:.3f}')

    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
print(f'\n✅ Best val_acc: {best_val_acc:.4f}')

## Step 6 — Export to ONNX

In [ ]:
import os

os.makedirs('models/doc_fraud', exist_ok=True)
model.eval().cpu()

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
onnx_path = 'models/doc_fraud/doc_fraud_detector.onnx'

torch.onnx.export(
    model, dummy, onnx_path,
    input_names=['image'],
    output_names=['logits'],
    dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=12,
    dynamo=False,   # force legacy exporter — no onnxscript needed
)

with open('models/doc_fraud/class_names.txt', 'w') as f:
    f.write('\n'.join(CLASSES))

size_mb = os.path.getsize(onnx_path) / 1024 / 1024
print(f'✅ Exported → {onnx_path}  ({size_mb:.1f} MB)')

## Step 7 — Download the model

This will download `doc_fraud_detector.onnx` to your computer.  
Place it in your project at: `models/doc_fraud/doc_fraud_detector.onnx`

In [ ]:
from google.colab import files
files.download('models/doc_fraud/doc_fraud_detector.onnx')
files.download('models/doc_fraud/class_names.txt')
print('✅ Download started! Place both files in your project at models/doc_fraud/')